# 🔬 SkinVision — Colab Training Launcher

This notebook is **strictly a launcher** for `training/train.py`. All ML modeling, dataset stratification, preprocessing, and evaluation logic are maintained within the modular codebase.

### Workflow:
1. Connect to a GPU runtime (**Runtime** > **Change runtime type** > **T4 GPU**).
2. Clone/pull the repository or mount Google Drive.
3. Download the HAM10000 dataset.
4. Execute `python training/train.py` with your chosen experiment configuration.
5. Download the trained `.keras` model and JSON metrics back to your local machine.

### 1. Verify GPU Acceleration

In [ ]:
!nvidia-smi

### 2. Setup Codebase & Dependencies
If your code is in a Git repository, clone it below. Alternatively, upload your workspace folder or mount Google Drive.

In [ ]:
import os

# If using git, uncomment and update your repo URL:
# !git clone <YOUR_REPO_URL> SkinVision
# %cd SkinVision

# Install exact pinned dependencies from the repository
!pip install -q -r requirements.txt
!pip install -q kagglehub

### 3. Download HAM10000 Dataset
Uses `kagglehub` to download and cache the HAM10000 dataset into your Colab VM environment.

In [ ]:
import kagglehub

print("Downloading HAM10000 dataset via kagglehub...")
dataset_path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("Dataset ready at:", dataset_path)

# Verify metadata file existence
metadata_file = os.path.join(dataset_path, 'HAM10000_metadata.csv')
if not os.path.exists(metadata_file):
    # Check if inside nested folder
    for root, dirs, files in os.walk(dataset_path):
        if 'HAM10000_metadata.csv' in files:
            dataset_path = root
            break

print("Using dataset root directory:", dataset_path)
assert os.path.exists(os.path.join(dataset_path, 'HAM10000_metadata.csv')), "Metadata CSV not found!"

### 4. Launch Experiment via `training/train.py`
You can customize hyperparameters in `training/config.yaml` or specify the experiment name below (e.g. `E0`, `E1`, `E2`, etc.).

In [ ]:
# Run experiment by ID (e.g. E1, E2, E3, etc.)
!python training/train.py \
    --data-dir "$dataset_path" \
    --config training/config.yaml \
    --experiment E2 \
    --notes "Extended fine-tuning: 80 layers, 25 epochs"

### 5. Download Model & Experiment Artifacts
Zips the full `experiments/` directory (containing per-experiment records, results_table.csv, and model checkpoints) as well as the canonical `models/` directory, and triggers a direct download to your local machine.

Once downloaded, unpack the contents into your local `experiments/` and `models/` directories.

In [ ]:
from google.colab import files
import shutil

# Archive the full experiments directory as well as models
shutil.make_archive('skinvision_experiments', 'zip', 'experiments')
shutil.make_archive('skinvision_models', 'zip', 'models')

# Download the archives directly to your computer
files.download('skinvision_experiments.zip')
files.download('skinvision_models.zip')
print("Download initiated! Extract the contents into your local SkinVision/ folder.")